# Data Preparation & Validation

This notebook loads and validates all raw datasets, builds master dataframes (human, model, BIO), and exports clean data for downstream analysis.

**Run this once at the start** before running other analysis notebooks.

**Outputs:**
- Validated human_master, model_master, bio_master dataframes
- Summary statistics and quality checks

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
import numpy as np
from src.data_loaders import load_human_master, load_model_master, load_bio_master
from src.config import get_paths

paths = get_paths()
print(f"✓ Data directory: {paths.data_dir}")
print(f"✓ Outputs directory: {paths.outputs_dir}")

✓ Data directory: C:\Users\AdamR\OneDrive\UCSB\VIU\HonorsThesis\data
✓ Outputs directory: C:\Users\AdamR\Projects\Flexible-Wisdom\outputs


## 1. Load Human Data

In [2]:
human_master = load_human_master()
print(f"✓ Loaded human data: {human_master.shape[0]} trials, {human_master['participantID'].nunique()} participants")
print(f"  Conditions: {sorted(human_master['condition'].unique())}")
print(f"  Columns: {list(human_master.columns)}")
human_master.head()

✓ Loaded human data: 36000 trials, 12 participants
  Conditions: ['100_0', '50_50', '80_20']
  Columns: ['stimID', 'condition', 'response', 'side_selected', 'cue_points', 'line1_angle', 'line2_angle', 'valid_cue', 'TP', 'participantID', 'decision']


,stimID,condition,response,side_selected,cue_points,line1_angle,line2_angle,valid_cue,TP,participantID,decision
0,100,50_50,6,1,2,14.314827,1.921956,False,True,SA,1
1,845,50_50,5,1,2,15.054317,4.222230,False,True,SA,1
2,245,50_50,4,1,1,14.314827,6.508956,True,True,SA,1
3,72,50_50,4,2,2,8.775056,15.054317,True,True,SA,1
4,469,50_50,4,2,2,4.222230,19.885165,True,True,SA,1


### Human Data Summary Statistics

In [3]:
print("Trials per participant:")
print(human_master.groupby('participantID').size().sort_values(ascending=False))
print("\nTrial outcome distribution (TP):")
print(human_master['TP'].value_counts().sort_index())
print("\nDecision distribution:")
print(human_master['decision'].value_counts().sort_index())

Trials per participant:
participantID
AG    3000
AW    3000
AZ    3000
BC    3000
CY    3000
GS    3000
HG    3000
JH    3000
KM    3000
KZ    3000
SA    3000
UR    3000
dtype: int64

Trial outcome distribution (TP):
TP
False    18000
True     18000
Name: count, dtype: int64

Decision distribution:
decision
0    16243
1    19757
Name: count, dtype: int64


## 2. Load Model Data

In [4]:
model_master = load_model_master()
print(f"✓ Loaded model data: {model_master.shape[0]} trials, {model_master['participantID'].nunique()} models")
print(f"  Models: {sorted(model_master['participantID'].unique())}")
print(f"  Columns: {list(model_master.columns)}")
model_master.head()

✓ Loaded model data: 36000 trials, 12 models
  Models: ['0responses_claude-3-5-haiku-20241022', '0responses_claude-3-7-sonnet-20250219', '0responses_claude-opus-4-20250514', '0responses_claude-sonnet-4-20250514', '0responses_gemini-2.5-flash', '0responses_gemini-2.5-pro', '0responses_gpt-4.1-2025-04-14', '0responses_gpt-5-2025-08-07', '0responses_gpt-5-mini-2025-08-07', '0responses_o3-2025-04-16', '0responses_o4-mini-2025-04-16', 'gemini-2.5-pro']
  Columns: ['stimID', 'condition', 'side_selected', 'cue_points', 'line1_angle', 'line2_angle', 'valid_cue', 'TP', 'response', 'participantID', 'decision']


,stimID,condition,side_selected,cue_points,line1_angle,line2_angle,valid_cue,TP,response,participantID,decision
0,100,50_50,1,2,14.314827,1.921956,False,True,present,0responses_claude-3-5-haiku-20241022,1
1,845,50_50,1,2,15.054317,4.222230,False,True,absent,0responses_claude-3-5-haiku-20241022,0
2,245,50_50,1,1,14.314827,6.508956,True,True,present,0responses_claude-3-5-haiku-20241022,1
3,72,50_50,2,2,8.775056,15.054317,True,True,absent,0responses_claude-3-5-haiku-20241022,0
4,469,50_50,2,2,4.222230,19.885165,True,True,absent,0responses_claude-3-5-haiku-20241022,0


### Model Data Summary Statistics

In [5]:
print("Trials per model:")
print(model_master.groupby('participantID').size().sort_values(ascending=False))
print("\nDecision distribution across models:")
print(model_master.groupby('participantID')['decision'].value_counts().sort_index())

Trials per model:
participantID
0responses_claude-3-5-haiku-20241022     3000
0responses_claude-3-7-sonnet-20250219    3000
0responses_claude-opus-4-20250514        3000
0responses_claude-sonnet-4-20250514      3000
0responses_gemini-2.5-flash              3000
0responses_gemini-2.5-pro                3000
0responses_gpt-4.1-2025-04-14            3000
0responses_gpt-5-2025-08-07              3000
0responses_gpt-5-mini-2025-08-07         3000
0responses_o3-2025-04-16                 3000
0responses_o4-mini-2025-04-16            3000
gemini-2.5-pro                           3000
dtype: int64

Decision distribution across models:
participantID                          decision
0responses_claude-3-5-haiku-20241022   0           1864
                                       1           1136
0responses_claude-3-7-sonnet-20250219  0           1603
                                       1           1397
0responses_claude-opus-4-20250514      0           1871
                                     

## 3. Load BIO (Bayesian Ideal Observer) Data

In [6]:
bio_master = load_bio_master()
print(f"✓ Loaded BIO data: {bio_master.shape[0]} trials")
print(f"  Columns: {list(bio_master.columns)}")
bio_master.head()

✓ Loaded BIO data: 3000 trials
  Columns: ['stimID', 'condition', 'side_selected', 'cue_points', 'line1_angle', 'line2_angle', 'valid_cue', 'TP', 'participantID', 'bio_sigma', 'bio_lambda', 'bio_llr', 'bio_p_present', 'bio_decision', 'bio_outcome']


,stimID,condition,side_selected,cue_points,line1_angle,line2_angle,valid_cue,TP,participantID,bio_sigma,bio_lambda,bio_llr,bio_p_present,bio_decision,bio_outcome
0,100,50_50,1,2,14.314827,1.921956,False,True,BIO,14.034651,6.348314,-1.515586,0.180112,0,0
1,845,50_50,1,2,15.054317,4.222230,False,True,BIO,14.034651,6.348314,-1.157873,0.239054,0,0
2,245,50_50,1,1,14.314827,6.508956,True,True,BIO,14.034651,6.348314,-0.601284,0.354050,0,0
3,72,50_50,2,2,8.775056,15.054317,True,True,BIO,14.034651,6.348314,-0.389093,0.403936,0,0
4,469,50_50,2,2,4.222230,19.885165,True,True,BIO,14.034651,6.348314,-2.420934,0.081590,0,0


### BIO Performance Summary

In [7]:
print(f"BIO decision distribution:")
print(bio_master['bio_decision'].value_counts().sort_index())
print(f"\nBIO accuracy: {bio_master['bio_outcome'].mean():.3f}")
print(f"BIO p_present (mean): {bio_master['bio_p_present'].mean():.3f}")

BIO decision distribution:
bio_decision
0    2871
1     129
Name: count, dtype: int64

BIO accuracy: 0.461
BIO p_present (mean): 0.337


## 4. Cross-Domain Consistency Checks

In [8]:
# Verify all domains have same stimIDs and conditions
human_stims = set(human_master['stimID'].unique())
model_stims = set(model_master['stimID'].unique())
bio_stims = set(bio_master['stimID'].unique())

print(f"✓ Human stimIDs: {len(human_stims)}")
print(f"✓ Model stimIDs: {len(model_stims)}")
print(f"✓ BIO stimIDs: {len(bio_stims)}")
print(f"✓ All match: {human_stims == model_stims == bio_stims}")

# Check condition alignment
print(f"\n✓ Conditions:")
print(f"  Human: {sorted(human_master['condition'].unique())}")
print(f"  Model: {sorted(model_master['condition'].unique())}")
print(f"  BIO: {sorted(bio_master['condition'].unique())}")

✓ Human stimIDs: 1000
✓ Model stimIDs: 1000
✓ BIO stimIDs: 1000
✓ All match: True

✓ Conditions:
  Human: ['100_0', '50_50', '80_20']
  Model: ['100_0', '50_50', '80_20']
  BIO: ['100_0', '50_50', '80_20']


## 5. Export Clean Data (Optional)
Save processed dataframes for reference or archival.

In [9]:
# Uncomment to export clean data
# human_master.to_csv(paths.outputs_dir / "human_master.csv", index=False)
# model_master.to_csv(paths.outputs_dir / "model_master.csv", index=False)
# bio_master.to_csv(paths.outputs_dir / "bio_master.csv", index=False)
# print("✓ Exported clean dataframes to outputs/")

print("✓ Data preparation complete. Ready for analysis.")

✓ Data preparation complete. Ready for analysis.
